Helper Functions

In [3]:
import os

az_dir = r"C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin"

os.environ["PATH"] = az_dir + os.pathsep + os.environ["PATH"]

import shutil
print("Azure CLI:", shutil.which("az"))

Azure CLI: C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.CMD


In [4]:
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

token = credential.get_token(
    "https://storage.azure.com/.default"
)

print("Authentication: OK")
print("Token expires:", token.expires_on)

Authentication: OK
Token expires: 1791216794


In [5]:
from deltalake import DeltaTable
from azure.identity import AzureCliCredential

path = (
    "abfss://4cecee03-0629-4081-8ae8-b749ee04d43f"
    "@onelake.dfs.fabric.microsoft.com/"
    "1ee46463-2495-44f5-9dc9-d635d0067483/"
    "Tables/Bronze/cursor_usage"
)

credential = AzureCliCredential()

token = credential.get_token(
    "https://storage.azure.com/.default"
)

storage_options = {
    "bearer_token": token.token,
    "use_fabric_endpoint": "true"
}

dt = DeltaTable(
    path,
    storage_options=storage_options
)

print("Delta table opened successfully")
print(dt.schema())

Delta table opened successfully
Schema([Field(Date, PrimitiveType("string"), nullable=True, metadata={'delta.columnMapping.physicalName': 'col-17a0e1f2-0dda-4758-b5ea-3bc7c9a37665', 'delta.columnMapping.id': 1}), Field(User, PrimitiveType("string"), nullable=True, metadata={'delta.columnMapping.physicalName': 'col-64404d8b-462a-480b-b102-12578914e3c0', 'delta.columnMapping.id': 2}), Field(Service Account Name, PrimitiveType("string"), nullable=True, metadata={'delta.columnMapping.physicalName': 'col-3eda8a01-0305-40ae-bcc4-32b234df40af', 'delta.columnMapping.id': 3}), Field(Service Account ID, PrimitiveType("string"), nullable=True, metadata={'delta.columnMapping.physicalName': 'col-53993f30-fd0d-47fe-8586-8a5057b49659', 'delta.columnMapping.id': 4}), Field(PR Author, PrimitiveType("string"), nullable=True, metadata={'delta.columnMapping.id': 5, 'delta.columnMapping.physicalName': 'col-654d05c0-1f08-449f-9235-fd034e08a5cc'}), Field(Repo, PrimitiveType("string"), nullable=True, metadata

In [3]:
%%sql
--select * from AI_Medallion_Bronze.dbo.HC_Bronze_Historical limit 5
--select * from AI_Medallion_Bronze.dbo.CoreOktaUser limit 5
--select * from AI_Medallion.Bronze.Cursor_CursorUsage limit 5



StatementMeta(, 8b6b1512-66b5-4637-bdad-96fe5c4376ed, 4, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 20 fields>

In [5]:
import os
import shutil
import subprocess

print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
print("java location =", shutil.which("java"))

try:
    result = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True
    )
    print("Java return code:", result.returncode)
    print(result.stderr)
except Exception as e:
    print("Java ERROR:", e)

JAVA_HOME = None
java location = None
Java ERROR: [WinError 2] The system cannot find the file specified


In [ ]:
from pyspark.sql.functions import (
    col, coalesce, lit, when, concat, trim, lower,
    year, month, to_date, to_timestamp, broadcast
)
from pyspark.sql.types import IntegerType, LongType, DecimalType
import re

CONTAINER = "4cecee03-0629-4081-8ae8-b749ee04d43f"
BRONZE_WS = "36e5350a-61f6-4120-8c32-a38f55722907"
SILVER_WS = "a1be6c30-3d8c-49ed-ace8-48930a92ba14"


def abfss(workspace, table):
    return (
        "abfss://" + CONTAINER
        + "@onelake.dfs.fabric.microsoft.com/"
        + workspace + "/Tables/dbo/" + table
    )


bronze_cursor_path = abfss(BRONZE_WS, "Cursor")
silver_cursor_path = abfss(SILVER_WS, "CursorUsage")
silver_okta_path   = abfss(SILVER_WS, "OktaUserforAI")
silver_hc_path     = abfss(SILVER_WS, "HC_Silver_Historical")

EMAIL_LIKE = "%_@_%._%"


def sanitise(name):
    return re.sub(r"[ ,;{}()\[\]\n\t=\\/.\-]+", "_", name).strip("_")


def delta_write(df, path):
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .save(path))


def safe_col(df, name, dtype="string"):
    """Return col(name) if it exists in df, otherwise lit(None) cast to dtype."""
    return col(name) if name in df.columns else lit(None).cast(dtype)


In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [2]:
df = spark.sql("SELECT * FROM AI_Medallion_Bronze.dbo.Cursor LIMIT 1000")
display(df)

NameError: name 'spark' is not defined